# SRA 実証実験 13: 仮想ニューロン単位での安全なホットスワップ（アンラーニング）

ノートブック12で存在が証明された「仮想ニューロン（Cell Assembly）」を単位として用いることで、シナプス単体では不可能だった**「他タスクへの破滅的忘却を伴わない、特定ドメイン知識のみの完全なアンラーニング（削除）」**と復元（挿入）が可能であることを実証します。

In [ ]:
import sys
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random
import numpy as np
import collections

if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch matplotlib seaborn numpy nbformat

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

from src.sra_gpu_models import MoESRAModel
from src.sra_experiment import make_optimizer, load_balance_loss, specialization_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. タスク設計とデータジェネレータ

5ドメイン × 5タスクを定義します。各ドメインは入力の文字分布が根本的に異なります。
モデルはキャラクターレベル（ASCII）で動作し、入力テキストを文字→ASCII整数にエンコードして処理します。

In [ ]:
import random, string

# ===== キャラクターレベル エンコーダー =====
VOCAB_SIZE = 128  # ASCII
PAD = 0
BOS = 1
EOS = 2

def encode(text):
    """ASCII文字列 → intリスト (BOS付き)"""
    return [BOS] + [ord(c) for c in text] + [EOS]

def decode(ids):
    """intリスト → 文字列 (制御トークン除外)"""
    return ''.join(chr(i) for i in ids if i >= 32)

def pad_to(seq, length):
    return seq[:length] + [PAD] * max(0, length - len(seq))

# ===== ドメイン1: 英語テキスト (NL) =====
WORDS = ["hello", "world", "python", "learn", "great", "smart", "open", "code",
         "data", "fast", "model", "brain", "build", "train", "green", "black"]

def nl_upper(w=None):
    w = w or random.choice(WORDS)
    return w, w.upper()

def nl_vowel_count(w=None):
    w = w or random.choice(WORDS)
    c = sum(1 for ch in w if ch in 'aeiou')
    return w, str(c)

def nl_reverse(w=None):
    w = w or random.choice(WORDS)
    return w, w[::-1]

def nl_length(w=None):
    w = w or random.choice(WORDS)
    return w, str(len(w))

def nl_first_char(w=None):
    w = w or random.choice(WORDS)
    return w, w[0]

# ===== ドメイン2: Pythonコード (CODE) =====
VARS = ["x", "y", "z", "n", "val", "res", "cnt", "idx"]
OPS  = ["+", "-", "*"]

def code_indent(v=None, op=None, n=None):
    v = v or random.choice(VARS); op = op or random.choice(OPS); n = n or random.randint(1,9)
    line = f"return {v} {op} {n}"
    return line, "    " + line

def code_is_comment():
    if random.random() < 0.5:
        line = f"# {random.choice(['add values','init','loop','check'])}"
        return line, "yes"
    else:
        v = random.choice(VARS); n = random.randint(1,9)
        return f"{v} = {n}", "no"

def code_operator(v=None, op=None, n=None):
    v = v or random.choice(VARS); op = op or random.choice(OPS); n = n or random.randint(1,9)
    return f"{v} = a {op} {n}", op

def code_varname(v=None, n=None):
    v = v or random.choice(VARS); n = n or random.randint(1,99)
    return f"{v} = {n}", v

def code_has_return():
    if random.random() < 0.5:
        v = random.choice(VARS)
        return f"return {v}", "yes"
    else:
        v = random.choice(VARS); n = random.randint(1,9)
        return f"{v} = {n}", "no"

# ===== ドメイン3: 数式 (MATH) =====
def math_add():
    a, b = random.randint(1,19), random.randint(1,19)
    return f"{a}+{b}=", str(a+b)

def math_sub():
    a, b = random.randint(1,19), random.randint(1,19)
    lo, hi = min(a,b), max(a,b)
    return f"{hi}-{lo}=", str(hi-lo)

def math_mul():
    a, b = random.randint(1,9), random.randint(1,9)
    return f"{a}*{b}=", str(a*b)

def math_gt():
    a, b = random.randint(1,19), random.randint(1,19)
    return f"{a}>{b}?", "yes" if a > b else "no"

def math_lt():
    a, b = random.randint(1,19), random.randint(1,19)
    return f"{a}<{b}?", "yes" if a < b else "no"

# ===== ドメイン4: DNA配列 (DNA) =====
BASES = ['A', 'T', 'G', 'C']
COMP  = {'A':'T', 'T':'A', 'G':'C', 'C':'G'}

def rand_dna(n=5):
    return ''.join(random.choices(BASES, k=n))

def dna_complement():
    s = rand_dna()
    return s, ''.join(COMP[c] for c in s)

def dna_reverse():
    s = rand_dna()
    return s, s[::-1]

def dna_repeat():
    s = rand_dna(3)
    return s, s * 2

def dna_count_g():
    s = rand_dna()
    return s, str(s.count('G'))

def dna_has_a():
    s = rand_dna()
    return s, "yes" if 'A' in s else "no"

# ===== ドメイン5: CSVデータ (CSV) =====
def rand_csv(n=4):
    nums = [random.randint(1, 20) for _ in range(n)]
    return ','.join(str(x) for x in nums), nums

def csv_count():
    s, nums = rand_csv()
    return s, str(len(nums))

def csv_max():
    s, nums = rand_csv()
    return s, str(max(nums))

def csv_min():
    s, nums = rand_csv()
    return s, str(min(nums))

def csv_first():
    s, nums = rand_csv()
    return s, str(nums[0])

def csv_last():
    s, nums = rand_csv()
    return s, str(nums[-1])

# ===== タスク辞書 =====
def make_sample(task_fn):
    result = task_fn()
    return result  # (input_str, output_str)

ALL_TASKS = {
    # NL
    "nl_upper":       lambda: nl_upper(),
    "nl_vowel_count": lambda: nl_vowel_count(),
    "nl_reverse":     lambda: nl_reverse(),
    "nl_length":      lambda: nl_length(),
    "nl_first_char":  lambda: nl_first_char(),
    # CODE
    "code_indent":    lambda: code_indent(),
    "code_is_comment":lambda: code_is_comment(),
    "code_operator":  lambda: code_operator(),
    "code_varname":   lambda: code_varname(),
    "code_has_return":lambda: code_has_return(),
    # MATH
    "math_add":       lambda: math_add(),
    "math_sub":       lambda: math_sub(),
    "math_mul":       lambda: math_mul(),
    "math_gt":        lambda: math_gt(),
    "math_lt":        lambda: math_lt(),
    # DNA
    "dna_complement": lambda: dna_complement(),
    "dna_reverse":    lambda: dna_reverse(),
    "dna_repeat":     lambda: dna_repeat(),
    "dna_count_g":    lambda: dna_count_g(),
    "dna_has_a":      lambda: dna_has_a(),
    # CSV
    "csv_count":      lambda: csv_count(),
    "csv_max":        lambda: csv_max(),
    "csv_min":        lambda: csv_min(),
    "csv_first":      lambda: csv_first(),
    "csv_last":       lambda: csv_last(),
}

DOMAIN_COLORS = {
    "nl":   "#4C72B0",
    "code": "#DD8452",
    "math": "#55A868",
    "dna":  "#C44E52",
    "csv":  "#8172B3",
}

def get_domain(task_name):
    return task_name.split('_')[0]

print(f"Total tasks: {len(ALL_TASKS)}")
for domain in ['nl','code','math','dna','csv']:
    tasks = [t for t in ALL_TASKS if get_domain(t)==domain]
    print(f"  {domain}: {tasks}")

# 動作確認
for name, fn in list(ALL_TASKS.items())[:5]:
    inp, out = fn()
    print(f"[{name}] '{inp}' → '{out}'")


## 2. モデル設定と学習

キャラクターレベル（ASCII 128）でSRAモデルを初期化し、25タスクを同時学習させます。
入力テキストの文字分布が根本的に異なるため、ルーターはタスクIDなしで自律的に各タスクを識別して別々のシナプスに振り分けるはずです。

In [ ]:
# モデル設定
MAX_SEQ_LEN = 24   # 最大入力文字数（入力+出力）
VOCAB_SIZE   = 128 # ASCII

DIM          = 64
LAYERS       = 2
NUM_SYNAPSES = 32
K            = 2
SYN_HIDDEN   = 128

model = MoESRAModel(
    vocab_size   = VOCAB_SIZE,
    dim          = DIM,
    layers       = LAYERS,
    num_synapses = NUM_SYNAPSES,
    k            = K,
    syn_hidden   = SYN_HIDDEN,
).to(device)
optimizer = make_optimizer(model, lr=1e-3)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Vocab size: {VOCAB_SIZE} (ASCII character-level)")
print(f"Synapses: {NUM_SYNAPSES}, k={K}")

def make_multitask_batch(tasks, batch_size, max_len=MAX_SEQ_LEN):
    """各タスクから均等にサンプルし、テキストをエンコードしてバッチ化する"""
    xs, ys = [], []
    task_names = list(tasks.keys())
    for _ in range(batch_size):
        task_name = random.choice(task_names)
        inp_str, out_str = tasks[task_name]()
        x = pad_to(encode(inp_str), max_len)
        y = pad_to(encode(out_str), max_len)
        xs.append(x)
        ys.append(y)
    x = torch.tensor(xs, dtype=torch.long, device=device)
    y = torch.tensor(ys, dtype=torch.long, device=device)
    return x, y


In [ ]:
# 学習ループ
print("Multitask Training on 25 Tasks started...")
model.train()

epochs     = 2000
batch_size = 128

for epoch in range(epochs):
    x, y = make_multitask_batch(ALL_TASKS, batch_size)
    
    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model(x, y_in)
    
    loss    = F.cross_entropy(outputs.reshape(-1, VOCAB_SIZE), y.reshape(-1), ignore_index=PAD)
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss
    
    total_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f} | LB Loss: {lb_loss.item():.4f}")

print("Training finished!")


In [ ]:
# 12の実験と同様に、学習済みモデルから仮想ニューロン（階層構造）を抽出する

import networkx as nx

def get_task_routing_vector(task_name, task_fn, samples=20):
    model.eval()
    synapse_counts = torch.zeros(NUM_SYNAPSES, device=device)
    total_valid_tokens = 0
    with torch.no_grad():
        for _ in range(samples):
            inp_str, out_str = task_fn()
            x       = torch.tensor([pad_to(encode(inp_str), MAX_SEQ_LEN)], dtype=torch.long, device=device)
            y_dummy = torch.tensor([pad_to(encode(out_str), MAX_SEQ_LEN)], dtype=torch.long, device=device)
            y_in    = torch.cat([torch.full((1, 1), BOS, dtype=torch.long, device=device), y_dummy[:, :-1]], dim=1)
            
            seq = torch.cat([x, y_in], dim=1)
            valid_mask = (seq != PAD)
            
            _, routing_logits, _ = model(x, y_in)
            logits = routing_logits[-1]
            _, topk_idx = torch.topk(logits, K, dim=-1)
            valid_topk = topk_idx[valid_mask]
            
            for k_idx in range(K):
                synapse_counts.index_add_(0, valid_topk[:, k_idx], torch.ones(valid_topk.size(0), device=device))
            total_valid_tokens += valid_topk.size(0)

    return (synapse_counts / total_valid_tokens).cpu().numpy()

task_vectors = {t_name: get_task_routing_vector(t_name, t_fn) for t_name, t_fn in ALL_TASKS.items()}

# Universal Cores (40%以上のタスクで使用) の抽出
UNIVERSAL_THRESHOLD = int(len(ALL_TASKS) * 0.4)
synapse_task_count = collections.defaultdict(int)
for vector in task_vectors.values():
    for syn in np.where(vector >= 0.15)[0]:
        synapse_task_count[syn] += 1
universal_cores = set(syn for syn, count in synapse_task_count.items() if count >= UNIVERSAL_THRESHOLD)

# Jaccard類似度によるクラスタリング
task_specialized = {t: set(np.where(vec >= 0.15)[0]) - universal_cores for t, vec in task_vectors.items()}
G = nx.Graph()
G.add_nodes_from(ALL_TASKS.keys())
tasks = list(ALL_TASKS.keys())
for i in range(len(tasks)):
    for j in range(i + 1, len(tasks)):
        set_a, set_b = task_specialized[tasks[i]], task_specialized[tasks[j]]
        if not set_a and not set_b: continue
        union = len(set_a | set_b)
        if union > 0 and (len(set_a & set_b) / union) >= 0.4:
            G.add_edge(tasks[i], tasks[j])

connected_components = sorted(list(nx.connected_components(G)), key=len, reverse=True)
print("仮想ニューロンの抽出完了！ (Connected Components extracted)")


In [ ]:
### ターゲットの選定と「安全なシナプスの特定」
# 独立性の高いDNAグループ（通常、構成タスクが5つのグループになる）をターゲットとします。

dna_comp = None
for comp in connected_components:
    if any(t.startswith("dna_") for t in comp):
        dna_comp = list(comp)
        break

if not dna_comp:
    dna_comp = [t for t in ALL_TASKS if t.startswith("dna_")]

print(f"Target Domain (Virtual Neuron): {dna_comp}")

# ターゲットが使用している全シナプス
dna_synapses = set()
for t in dna_comp:
    dna_synapses.update(np.where(task_vectors[t] >= 0.15)[0])

# それ以外の全タスクが使用しているシナプス
other_synapses = set()
for t in ALL_TASKS.keys():
    if t not in dna_comp:
        other_synapses.update(np.where(task_vectors[t] >= 0.15)[0])

# 安全に削除できるシナプス（参照カウントが1＝DNAしか使っていないシナプス）
safe_to_delete = dna_synapses - other_synapses

print(f"Target Group Synapses : {[int(x) for x in sorted(list(dna_synapses))]}")
print(f"Other Tasks Synapses  : {[int(x) for x in sorted(list(other_synapses))]}")
print(f"Safe to Delete (RC=1) : {[int(x) for x in sorted(list(safe_to_delete))]}")

if len(safe_to_delete) == 0:
    print("Warning: DNA専用のシナプスが存在しません。DNAの学習が他のタスクと完全に融合しています。")


In [ ]:
### 精度計測関数（ベースライン計測）

def evaluate_domain(domain_tasks, samples=50):
    model.eval()
    total_acc = 0
    with torch.no_grad():
        for t_name in domain_tasks:
            t_fn = ALL_TASKS[t_name]
            accs = []
            for _ in range(samples):
                inp_str, out_str = t_fn()
                x = torch.tensor([pad_to(encode(inp_str), MAX_SEQ_LEN)], dtype=torch.long, device=device)
                y = torch.tensor([pad_to(encode(out_str), MAX_SEQ_LEN)], dtype=torch.long, device=device)
                y_in = torch.cat([torch.full((1, 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
                
                logits, _, _ = model(x, y_in)
                preds = logits.argmax(dim=-1)
                
                mask = (y != PAD)
                if mask.sum() > 0:
                    acc = (preds[mask] == y[mask]).float().mean().item()
                    accs.append(acc)
            total_acc += np.mean(accs)
    return total_acc / len(domain_tasks)

other_tasks = [t for t in ALL_TASKS.keys() if t not in dna_comp]

print("=== 削除前の精度（ベースライン） ===")
dna_acc_before = evaluate_domain(dna_comp)
other_acc_before = evaluate_domain(other_tasks)
print(f"DNA Tasks Accuracy  : {dna_acc_before*100:.1f}%")
print(f"Other Tasks Accuracy: {other_acc_before*100:.1f}%")


In [ ]:
### 仮想ニューロンの削除（物理的なアンラーニング）

import copy
# 復元用にバックアップを取る
backup_state = copy.deepcopy(model.state_dict())

def delete_synapses(synapses_to_delete):
    with torch.no_grad():
        for layer in model.layers:
            block = layer.moe
            for idx in synapses_to_delete:
                # 該当するシナプス（エキスパート）の重みを物理的にゼロ化する
                block.w1.data[idx].zero_()
                block.b1.data[idx].zero_()
                block.w2.data[idx].zero_()

if len(safe_to_delete) > 0:
    delete_synapses(safe_to_delete)
    print(f"Synapses {[int(x) for x in sorted(list(safe_to_delete))]} の重みを物理的に破壊（ゼロ化）しました。")
else:
    print("削除対象の専用シナプスがないため、破壊をスキップします。")

print("\n=== 削除後（アンラーニング）の精度 ===")
dna_acc_after = evaluate_domain(dna_comp)
other_acc_after = evaluate_domain(other_tasks)
print(f"DNA Tasks Accuracy  : {dna_acc_after*100:.1f}%  <-- (Expected: 大きく低下)")
print(f"Other Tasks Accuracy: {other_acc_after*100:.1f}%  <-- (Expected: 維持される)")


In [ ]:
### 仮想ニューロンの復元（ホットスワップ挿入）

# バックアップから重みを復元
model.load_state_dict(backup_state)
print("バックアップ（知識カプセル）からシナプスを再ロードしました。")

print("\n=== 復元後（ホットスワップ挿入）の精度 ===")
dna_acc_restored = evaluate_domain(dna_comp)
other_acc_restored = evaluate_domain(other_tasks)
print(f"DNA Tasks Accuracy  : {dna_acc_restored*100:.1f}%  <-- (Expected: 元の精度に復活)")
print(f"Other Tasks Accuracy: {other_acc_restored*100:.1f}%  <-- (Expected: 維持される)")
